In [40]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [42]:
# ---------------------------------------------------------
# 1. Datos del problema
# ---------------------------------------------------------
E = np.array([0.065, 0.040, 0.050])
Omega = np.array([
    [0.02560, 0.00392, 0.00384],
    [0.00392, 0.00490, 0.00168],
    [0.00384, 0.00168, 0.01440]
])
ones = np.ones(3)
Omega_inv = np.linalg.inv(Omega)

# Sistema de Markowitz
a11 = float(E.T @ Omega_inv @ E)
a12 = float(ones.T @ Omega_inv @ E)
a22 = float(ones.T @ Omega_inv @ ones)
A = np.array([[a11, a12], [a12, a22]])
delta = np.linalg.det(A)

# Definimos un retorno objetivo mu (ejemplo: 5.5%)
mu = 0.055

# Multiplicadores de Lagrange
lambda1 = (a22 * mu - a12) / delta
lambda2 = (a11 - a12 * mu) / delta

# Portafolio óptimo w*
w_opt = lambda1 * (Omega_inv @ E) + lambda2 * (Omega_inv @ ones)
var_opt = float(w_opt.T @ Omega @ w_opt)

# Gradientes en el punto óptimo
grad_f = Omega @ w_opt
grad_g1 = E
grad_g2 = ones

In [43]:
# ---------------------------------------------------------
# 2. Construcción de geometrías 3D
# ---------------------------------------------------------
# Malla para los planos de restricción
w1_grid, w2_grid = np.meshgrid(np.linspace(0.1, 0.8, 30), np.linspace(-0.1, 0.5, 30))

# Plano 1: Presupuesto (w1 + w2 + w3 = 1 -> w3 = 1 - w1 - w2)
W3_budget = 1 - w1_grid - w2_grid

# Plano 2: Retorno (E1*w1 + E2*w2 + E3*w3 = mu -> w3 = (mu - E1*w1 - E2*w2) / E3)
W3_return = (mu - E[0] * w1_grid - E[1] * w2_grid) / E[2]

# Recta de intersección (Portafolios Factibles)
dir_vector = np.cross(ones, E)
dir_vector = dir_vector / np.linalg.norm(dir_vector)
t = np.linspace(-0.4, 0.4, 100)
line_pts = w_opt[:, None] + dir_vector[:, None] * t

# Elipsoide de varianza mínima (w^T Omega w = var_opt)
evals, evecs = np.linalg.eigh(Omega)
u = np.linspace(0, 2 * np.pi, 40)
v = np.linspace(0, np.pi, 40)
x_s = np.outer(np.cos(u), np.sin(v))
y_s = np.outer(np.sin(u), np.sin(v))
z_s = np.outer(np.ones_like(u), np.cos(v))

sphere = np.stack([x_s, y_s, z_s], axis=-1)
scaled_sphere = sphere * np.sqrt(var_opt / evals)
ellipsoid = np.einsum('ij,uvj->uvi', evecs, scaled_sphere)

# ---------------------------------------------------------
# Plano normal equilibrado (Construcción gráfica invisible por Gram-Schmidt)
# ---------------------------------------------------------
# Para evitar que el plano se dibuje como un rombo/aguja picudo (debido a que ∇g1 y ∇g2 
# son casi paralelos), construimos internamente un vector auxiliar u2_ortho mediante Gram-Schmidt.
# Este vector es 100% invisible en la gráfica (no se le añade ningún trace) y solo sirve
# como eje ortogonal interno para parametrizar la malla de Plotly como un cuadrado simétrico.

u1_ortho = grad_g1 / np.linalg.norm(grad_g1)
u2_ortho = grad_g2 - np.dot(grad_g2, u1_ortho) * u1_ortho  # Vector ortogonal auxiliar
u2_ortho = u2_ortho / np.linalg.norm(u2_ortho)             # Base ortonormal estricta del plano

span = 0.4  # Dimensión del cuadrado alrededor de w*
u_span = np.linspace(-span, span, 20)
v_span = np.linspace(-span, span, 20)
U_grid, V_grid = np.meshgrid(u_span, v_span)

X_span = w_opt[0] + U_grid * u1_ortho[0] + V_grid * u2_ortho[0]
Y_span = w_opt[1] + U_grid * u1_ortho[1] + V_grid * u2_ortho[1]
Z_span = w_opt[2] + U_grid * u1_ortho[2] + V_grid * u2_ortho[2]

In [44]:
# ---------------------------------------------------------
# 3. Gráfica interactiva Plotly
# ---------------------------------------------------------
fig = go.Figure()

# Superficie 1: Plano de Presupuesto
fig.add_trace(go.Surface(
    x=w1_grid, y=w2_grid, z=W3_budget,
    colorscale='Blues', opacity=0.35, showscale=False,
    name='Plano Presupuesto (∑w = 1.0)'
))

# Superficie 2: Plano de Retorno (Incluye mu en la leyenda)
fig.add_trace(go.Surface(
    x=w1_grid, y=w2_grid, z=W3_return,
    colorscale='Oranges', opacity=0.35, showscale=False,
    name=f'Plano Retorno Objetivo (μ = {mu:.3f})'
))

# Superficie 3: Elipsoide de Varianza Mínima (Incluye var_opt en la leyenda)
fig.add_trace(go.Surface(
    x=ellipsoid[:, :, 0], y=ellipsoid[:, :, 1], z=ellipsoid[:, :, 2],
    colorscale='Viridis', opacity=0.3, showscale=False,
    name=f'Elipsoide Nivel Varianza (σ² = {var_opt:.6f})'
))

# Superficie 4: Plano Normal Cuadrado Equilibrado
fig.add_trace(go.Surface(
    x=X_span, y=Y_span, z=Z_span,
    colorscale='Purples', opacity=0.45, showscale=False,
    name='Plano Generado por ∇g1 y ∇g2'
))

# Recta de intersección de las restricciones
fig.add_trace(go.Scatter3d(
    x=line_pts[0], y=line_pts[1], z=line_pts[2],
    mode='lines', line=dict(color='black', width=6),
    name='Portafolios Factibles (Intersección)'
))

# Punto Óptimo w*
fig.add_trace(go.Scatter3d(
    x=[w_opt[0]], y=[w_opt[1]], z=[w_opt[2]],
    mode='markers', marker=dict(size=8, color='red', symbol='diamond'),
    name=f'Punto Óptimo w* ({w_opt[0]:.2f}, {w_opt[1]:.2f}, {w_opt[2]:.2f})'
))

# Normalización visible de los 3 gradientes originales a una longitud fija
arrow_len = 0.25
n_grad_f = (grad_f / np.linalg.norm(grad_f)) * arrow_len
n_grad_g1 = (grad_g1 / np.linalg.norm(grad_g1)) * arrow_len
n_grad_g2 = (grad_g2 / np.linalg.norm(grad_g2)) * arrow_len

# Vector Gradiente de f
fig.add_trace(go.Scatter3d(
    x=[w_opt[0], w_opt[0] + n_grad_f[0]],
    y=[w_opt[1], w_opt[1] + n_grad_f[1]],
    z=[w_opt[2], w_opt[2] + n_grad_f[2]],
    mode='lines+markers', line=dict(color='purple', width=6),
    marker=dict(size=4),
    name='∇f(w*) Gradiente Varianza'
))

# Vector Gradiente de g1 (Retorno)
fig.add_trace(go.Scatter3d(
    x=[w_opt[0], w_opt[0] + n_grad_g1[0]],
    y=[w_opt[1], w_opt[1] + n_grad_g1[1]],
    z=[w_opt[2], w_opt[2] + n_grad_g1[2]],
    mode='lines+markers', line=dict(color='darkorange', width=5),
    marker=dict(size=4),
    name='∇g1 (Normal Retorno E)'
))

# Vector Gradiente de g2 (Presupuesto)
fig.add_trace(go.Scatter3d(
    x=[w_opt[0], w_opt[0] + n_grad_g2[0]],
    y=[w_opt[1], w_opt[1] + n_grad_g2[1]],
    z=[w_opt[2], w_opt[2] + n_grad_g2[2]],
    mode='lines+markers', line=dict(color='royalblue', width=5),
    marker=dict(size=4),
    name='∇g2 (Normal Presupuesto 1)'
))

# Configuración de layout con información explicativa en el título
fig.update_layout(
    title=f'<b>Geometría 3D de Markowitz con 2 Restricciones</b><br>'
          f'<sup>Retorno Objetivo μ = {mu:.3f} | Varianza Mínima σ² = {var_opt:.6f} (Volatilidad σ = {np.sqrt(var_opt):.4f})</sup>',
    scene=dict(
        xaxis_title='w1 (Stocks)',
        yaxis_title='w2 (Bonds)',
        zaxis_title='w3 (Real Estate)',
        aspectmode='data'
    ),
    width=950,
    height=800
)

fig.show()